In [1]:
import pandas as pd
import numpy as np

- keep only the regions where the weather stations are located above 1400m

|  station_name |  avalanche_region_id |
|---|---|
| Galtür  | AT-07-10  |
|  Hahnenkamm-Ehrenbachhöhe | AT-07-17  |
|  Patscherkofel | AT-07-15  |
|  Valluga |  AT-07-07 |
|  Schmirn |  AT-07-23 |
|  Obergurgl |  AT-07-21 |
|  Pitztaler Gletscher | AT-07-20  |
| St. Leonhard im Pitztal  | AT-07-14  |
| Ischgl-Idalpe  | AT-07-13  |


In [2]:
data = pd.read_csv("data/combined_cleaned.csv")

KEEP_REGIONS = ['AT-07-10', 'AT-07-17', 'AT-07-15', 'AT-07-07', 'AT-07-23',
                'AT-07-20', 'AT-07-21', 'AT-07-14', 'AT-07-13']

data = data[data["region"].isin(KEEP_REGIONS)].copy()

data["date"] = pd.to_datetime(data["date"])
data["published"] = (pd.to_datetime(data["published"], utc=True)
                       .dt.tz_convert("Europe/Vienna"))

data = data.sort_values(["region", "date", "published"]).reset_index(drop=True)

pub_day = data["published"].dt.tz_localize(None).dt.normalize()
data["days_before"] = (data["date"] - pub_day).dt.days
print(data["days_before"].value_counts().sort_index())
data.info()

days_before
0     838
1    7533
Name: count, dtype: int64
<class 'pandas.DataFrame'>
RangeIndex: 8371 entries, 0 to 8370
Data columns (total 30 columns):
 #   Column             Non-Null Count  Dtype                        
---  ------             --------------  -----                        
 0   date               8371 non-null   datetime64[us]               
 1   season             8371 non-null   str                          
 2   region             8371 non-null   str                          
 3   region_name        8371 non-null   str                          
 4   status             8371 non-null   str                          
 5   danger_max         8371 non-null   int64                        
 6   danger_above       8371 non-null   int64                        
 7   danger_below       8299 non-null   float64                      
 8   elev_limit         6279 non-null   str                          
 9   danger_am          8371 non-null   int64                        
 10  d

In [3]:
# check for duplicates in (region, date) and drop them, keeping the last published
duplicates = data.duplicated(["region", "date"]).sum()
print(f"duplicate (region, date) rows: {duplicates}")
data = data.drop_duplicates(["region", "date"], keep="last").reset_index(drop=True)

# sanity checks for snow depth, humidity, precipitation, and wind gusts
print("snow_depth <0:", (data["snow_depth_cm"] < 0).sum(),
      "| humidity outside 0-100:", (~data["rel_humidity_mean"].between(0, 100)).sum(),
      "| precip <0:", (data["precip_sum_mm"] < 0).sum(),
      "| gust < mean wind:", (data["wind_gust_max"] < data["wind_speed_mean"]).sum())
data["snow_depth_cm"] = data["snow_depth_cm"].clip(lower=0)
data["precip_sum_mm"] = data["precip_sum_mm"].clip(lower=0)
data["rel_humidity_mean"] = data["rel_humidity_mean"].clip(0, 100)


check = np.fmax(data["danger_above"], data["danger_below"].fillna(0))
print("danger_max != max(above, below):", (check != data["danger_max"]).sum())
print(data["danger_max"].value_counts().sort_index())

data.head(10)
data.info()

duplicate (region, date) rows: 0
snow_depth <0: 0 | humidity outside 0-100: 0 | precip <0: 0 | gust < mean wind: 0
danger_max != max(above, below): 0
danger_max
1    1086
2    4207
3    2696
4     382
Name: count, dtype: int64
<class 'pandas.DataFrame'>
RangeIndex: 8371 entries, 0 to 8370
Data columns (total 30 columns):
 #   Column             Non-Null Count  Dtype                        
---  ------             --------------  -----                        
 0   date               8371 non-null   datetime64[us]               
 1   season             8371 non-null   str                          
 2   region             8371 non-null   str                          
 3   region_name        8371 non-null   str                          
 4   status             8371 non-null   str                          
 5   danger_max         8371 non-null   int64                        
 6   danger_above       8371 non-null   int64                        
 7   danger_below       8299 non-null   float64

In [4]:
data = data.drop(columns="status") # can be dropped, was just for downloading the data from the API and checking if the data is complete
data = data.drop(columns="danger_patterns") # dropping since we are not planning to use them in the future
print(data["region_name"].nunique())

print(data.groupby(["region", "region_name"])
          .agg(rows=("date", "size"), first=("date", "min"), last=("date", "max")))

12
                                                 rows      first       last
region   region_name                                                       
AT-07-07 Lechtaler Alpen West                     154 2025-11-29 2026-05-01
         Westliche Lechtaler Alpen                927 2019-11-16 2025-05-01
AT-07-10 Verwallgruppe Mitte                      154 2025-11-29 2026-05-01
         Westliche Verwallgruppe                  927 2019-11-16 2025-05-01
AT-07-13 Samnaungruppe                           1081 2019-11-16 2026-05-01
AT-07-14 Nördliche Ötztaler- und Stubaier Alpen   479 2019-11-16 2022-05-01
AT-07-15 Tuxer Alpen West                         154 2025-11-29 2026-05-01
         Westliche Tuxer Alpen                    927 2019-11-16 2025-05-01
AT-07-17 Westliche Kitzbüheler Alpen              479 2019-11-16 2022-05-01
AT-07-20 Weißkugelgruppe                         1081 2019-11-16 2026-05-01
AT-07-21 Gurgler Gruppe                          1081 2019-11-16 2026-05-01
AT-07-23 



- the region names changed over time

| ID  |  old name (2019-2025) | new name (>29.11.2025)   |
|---|---|---|
| AT-07-07  |  Western Lechtal Alps  | Lechtal Alps West  |
|  AT-07-10 | Western Verwallgroup  |  Verwallgroup Middle |
|AT-07-15| Western Tuxeralps  |  Tuxer Alps West |


In [5]:
latest_name = (data.sort_values("date")
                   .groupby("region")["region_name"]
                   .last())
data["region_name"] = data["region"].map(latest_name)

print(data["region"].nunique(), data["region_name"].nunique())

print(data.groupby(["region", "region_name"])
          .agg(rows=("date", "size"), first=("date", "min"), last=("date", "max")))

9 9
                                                 rows      first       last
region   region_name                                                       
AT-07-07 Lechtaler Alpen West                    1081 2019-11-16 2026-05-01
AT-07-10 Verwallgruppe Mitte                     1081 2019-11-16 2026-05-01
AT-07-13 Samnaungruppe                           1081 2019-11-16 2026-05-01
AT-07-14 Nördliche Ötztaler- und Stubaier Alpen   479 2019-11-16 2022-05-01
AT-07-15 Tuxer Alpen West                        1081 2019-11-16 2026-05-01
AT-07-17 Westliche Kitzbüheler Alpen              479 2019-11-16 2022-05-01
AT-07-20 Weißkugelgruppe                         1081 2019-11-16 2026-05-01
AT-07-21 Gurgler Gruppe                          1081 2019-11-16 2026-05-01
AT-07-23 Nördliche Zillertaler Alpen              927 2019-11-16 2025-05-01


- add the weather station names

In [6]:
station_names = {
    "AT-07-10": "Galtür",
    "AT-07-17": "Hahnenkamm-Ehrenbachhöhe",
    "AT-07-15": "Patscherkofel",
    "AT-07-07": "Valluga",
    "AT-07-23": "Schmirn",
    "AT-07-21": "Obergurgl",
    "AT-07-20": "Pitztaler Gletscher",
    "AT-07-14": "St. Leonhard im Pitztal",
    "AT-07-13": "Ischgl-Idalpe",
}

data["station_name"] = data["region"].map(station_names)

data.head(20)
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 8371 entries, 0 to 8370
Data columns (total 29 columns):
 #   Column             Non-Null Count  Dtype                        
---  ------             --------------  -----                        
 0   date               8371 non-null   datetime64[us]               
 1   season             8371 non-null   str                          
 2   region             8371 non-null   str                          
 3   region_name        8371 non-null   str                          
 4   danger_max         8371 non-null   int64                        
 5   danger_above       8371 non-null   int64                        
 6   danger_below       8299 non-null   float64                      
 7   elev_limit         6279 non-null   str                          
 8   danger_am          8371 non-null   int64                        
 9   danger_pm          8371 non-null   int64                        
 10  problems           7893 non-null   str                     

- missing 2092 elevation_limit values, because the danger value is basically the same across all elevations, so we are filling the missing values with a new string "no_elevation_limit" to indicate that there is no elevation limit for the danger value
- missing 478 specific avalanche problems, replacing nan with "no_avalanche_problem" to indicate that there is no specific avalanche problem for the danger value
- still missing 72 danger_below values, which indicate the danger below the elevation limit, we are not filling these values, since we have no information about to fill the out this info

In [7]:
data["elev_limit"] = data["elev_limit"].fillna("no_elev_limit")


print(data["problems"].unique())
data["problems"] = data["problems"].fillna("no_avalanche_problem")

data.info()
data.to_csv("data/final_avalanche_data.csv", index=False)

<StringArray>
[                                'gliding_snow;wind_slab',
                                     'wet_snow;wind_slab',
                                              'wind_slab',
                                     'new_snow;wind_slab',
                       'persistent_weak_layers;wind_slab',
                    'gliding_snow;persistent_weak_layers',
                                           'gliding_snow',
                                   'favourable_situation',
                                                      nan,
                                               'new_snow',
                        'new_snow;persistent_weak_layers',
                        'persistent_weak_layers;wet_snow',
              'persistent_weak_layers;wet_snow;wind_slab',
                        'gliding_snow;wet_snow;wind_slab',
                                  'gliding_snow;wet_snow',
             'favourable_situation;gliding_snow;wet_snow',
                                      'new